# Cleaning and merging datasets

The goal of this notebook is to **merge the data** collected from API with the consolidated historical data obtained by running create_complete_hist_dataset.ipynb. Then, we **clean the data**, make sure the data is **qualitative** enough and plot the first graphs to have an **overview of the cleaned dataset**.

## Importations

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import folium
from folium.plugins import MarkerCluster
import ast

#For ZTL printed on map
import requests
import json

#For segmented datasets that we will analyse
from shapely.geometry import Point, shape
from shapely.ops import unary_union
from geopy.distance import geodesic # Pour un calcul précis en mètres

In [11]:
# --- Loading historical data ---
path_historical_data = '/Users/quentingirard/Downloads/MCNDU_sorted_historical_dataset.csv'

if not os.path.exists(path_historical_data):
    raise FileNotFoundError(f"The historical file is not found at: {path_historical_data}")
hist_df = pd.read_csv(path_historical_data, sep=";", encoding="utf-8")

In [12]:
# --- Loading API data ---

### Must adapt this code for all the paths, as there is one file every 10 days

path_api_data_1 = '/Users/quentingirard/Desktop/Ecole/3A_Ponts/MCNDU/Projet-comptage-routier/petit-entrepot-pour-MCNDU/data/traffic_2025-12_1.parquet'
path_api_data_2 = '/Users/quentingirard/Desktop/Ecole/3A_Ponts/MCNDU/Projet-comptage-routier/petit-entrepot-pour-MCNDU/data/traffic_2025-12_2.parquet'
path_api_data_3 = '/Users/quentingirard/Desktop/Ecole/3A_Ponts/MCNDU/Projet-comptage-routier/petit-entrepot-pour-MCNDU/data/traffic_2025-12_3.parquet'
path_api_data_4 = '/Users/quentingirard/Desktop/Ecole/3A_Ponts/MCNDU/Projet-comptage-routier/petit-entrepot-pour-MCNDU/data/traffic_2026-01_1.parquet'

if not os.path.exists(path_api_data_1):
    raise FileNotFoundError(f"The API file is not found at: {path_api_data_1}")
api_df_1 = pd.read_parquet(path_api_data_1, engine='pyarrow')
    
if not os.path.exists(path_api_data_2):
    raise FileNotFoundError(f"The API file is not found at: {path_api_data_2}")
api_df_2 = pd.read_parquet(path_api_data_2, engine='pyarrow')

if not os.path.exists(path_api_data_3):
    raise FileNotFoundError(f"The API file is not found at: {path_api_data_3}")
api_df_3 = pd.read_parquet(path_api_data_3, engine='pyarrow')

if not os.path.exists(path_api_data_4):
    raise FileNotFoundError(f"The API file is not found at: {path_api_data_4}")
api_df_4 = pd.read_parquet(path_api_data_4, engine='pyarrow')

### Merging the four API datasets
api_df = pd.concat([api_df_1, api_df_2, api_df_3, api_df_4], ignore_index=True)
api_df = api_df.reset_index(drop=True)

## Cleaning raw data

**STEP 1:** Renaming, changing date format, droppping datapoints without position, extracting longitude and latitude

In [13]:
# --- 0 Useful functions ---

def get_lat_from_list(x):
    # Check if x is a list (or an array) and if it has at least 2 elements
    if isinstance(x, (list, np.ndarray)) and len(x) >= 2:
        return x[0]
    return None

def get_lon_from_list(x):
    if isinstance(x, (list, np.ndarray)) and len(x) >= 2:
        return x[1]
    return None

# --- 1. Historical Dataset ---
mapping_hist = {
    'Identifiant arc': 'Arc ID',
    'Libelle': 'Arc Name',
    'Date et heure de comptage': 'Datetime',
    'Débit horaire': 'Debit',
    'Taux d\'occupation': 'Occupation',
    'Etat trafic': 'Trafic',
    'Identifiant noeud amont': 'ID Upstream Node',
    'Libelle noeud amont': 'Upstream Node Name',
    'Identifiant noeud aval': 'ID Downstream Node',
    'Libelle noeud aval': 'Downstream Node Name',
    'Etat arc': 'Arc State',
    'Date debut dispo data': 'Start Date Available Data',
    'Date fin dispo data': 'End Date Available Data'
}

hist_df = hist_df.rename(columns=mapping_hist)

hist_df["Datetime"] = pd.to_datetime(hist_df["Datetime"], utc=True)

hist_df = hist_df.dropna(subset=['geo_point_2d']) #droping None values for geo_point_2d

# 2. [CORRECTION OPTIMISÉE] Séparation vectorielle
# On s'assure que c'est bien du string, puis on coupe à la virgule
print("Extraction rapide des coordonnées (Vectorisée)...")

# Cela sépare "48.86, 2.31" en deux colonnes distinctes instantanément
# expand=True crée un DataFrame de 2 colonnes
coords_split = hist_df['geo_point_2d'].astype(str).str.split(',', expand=True)

# 3. Conversion en float et assignation
# La colonne 0 est la Latitude, la 1 est la Longitude
# pd.to_numeric avec errors='coerce' gère les cas où la conversion échoue (devient NaN)
hist_df['Latitude'] = pd.to_numeric(coords_split[0], errors='coerce')
hist_df['Longitude'] = pd.to_numeric(coords_split[1], errors='coerce')

# 4. Vérification finale
missing_coords = hist_df['Latitude'].isna().sum()
print(f"Après correction vectorisée, nombre de lignes sans coordonnées : {missing_coords} (sur {len(hist_df)})")

# Nettoyage
hist_df.drop(columns=['geo_point_2d', 'geo_shape'], inplace=True, errors='ignore')

# --- 2. API Dataset ---

columns_mapping = {
    'iu_ac': 'Arc ID',
    'libelle': 'Arc Name',
    't_1h': 'Datetime',
    'q': 'Debit',               # q = débit (flow)
    'k': 'Occupation',          # k = taux d'occupation
    'etat_trafic': 'Trafic',
    'iu_nd_amont': 'ID Upstream Node',
    'libelle_nd_amont': 'Upstream Node Name',
    'iu_nd_aval': 'ID Downstream Node',
    'libelle_nd_aval': 'Downstream Node Name',
    'etat_barre': 'Arc State',
    'date_debut': 'Start Date Available Data',
    'date_fin': 'End Date Available Data'
}

api_df = api_df.rename(columns=columns_mapping)

api_df["Datetime"] = pd.to_datetime(api_df["Datetime"], utc=True)

api_df = api_df.dropna(subset=['geo_point_2d']) #droping None values

api_df['Latitude'] = api_df['geo_point_2d'].apply(get_lat_from_list)
api_df['Longitude'] = api_df['geo_point_2d'].apply(get_lon_from_list)

api_df.drop(columns=['geo_point_2d', 'geo_shape'], inplace=True, errors='ignore')

# Sorting by decreasing date (we don't do it for historical dataset as it is already sorted from create_complete_hist_dataset.ipynb)
api_df = api_df.sort_values(by="Datetime", ascending=False)

# --- 3. Verification ---
# hist_df.head()
# api_df.head()

Extraction rapide des coordonnées (Vectorisée)...
Après correction vectorisée, nombre de lignes sans coordonnées : 0 (sur 28060164)


**STEP 2:** Getting relevant datatypes for each feature

In [14]:
# --- 1. Historical Dataset ---

# Date Conversion
date_cols = ['End Date Available Data', 'Start Date Available Data']

for col in date_cols:
    hist_df[col] = pd.to_datetime(hist_df[col], errors='coerce')

# Integer Conversion
int_cols = ['Arc ID', 'ID Upstream Node', 'ID Downstream Node']

for col in int_cols:
    # Step 1: Force numeric conversion, errors become NaN
    hist_df[col] = pd.to_numeric(hist_df[col], errors='coerce')
    # Step 2: Convert to Integer type compatible with NaN
    hist_df[col] = hist_df[col].astype('Int64')

# String Conversion
str_cols = ['Arc Name', 'Trafic', 'Upstream Node Name', 'Downstream Node Name']

for col in str_cols:
    hist_df[col] = hist_df[col].astype('string')

# Binary Variables
mapping_state = {
    'Ouvert': 1,
    'Fermé': 0
}

hist_df['Arc State'] = hist_df['Arc State'].map(mapping_state)
hist_df['Arc State'] = hist_df['Arc State'].astype('Int64')

# --- 2. API Dataset ---

# Date conversion
date_cols = ['End Date Available Data', 'Start Date Available Data']

for col in date_cols:
    api_df[col] = pd.to_datetime(api_df[col], errors='coerce')

# Integer conversion
int_cols = ['Arc ID', 'ID Upstream Node', 'ID Downstream Node']

for col in int_cols:
    # Step 1: Force numeric conversion, errors become NaN
    api_df[col] = pd.to_numeric(api_df[col], errors='coerce')
    # Step 2: Convert to Integer type compatible with NaN
    api_df[col] = api_df[col].astype('Int64')

# String conversion
str_cols = ['Arc Name', 'Trafic', 'Upstream Node Name', 'Downstream Node Name']

for col in str_cols:
    api_df[col] = api_df[col].astype('string')

# Binary variables
mapping_state = {
    'Ouvert': 1,
    'Fermé': 0
}

api_df['Arc State'] = api_df['Arc State'].map(mapping_state)

api_df['Arc State'] = api_df['Arc State'].astype('Int64')

# --- 3. Verification ---
# print(hist_df.info())
# print(api_df.info())

## Merging the datasets

In [15]:
# We put api_df first in the list so it appears "on top"
full_df = pd.concat([api_df, hist_df], ignore_index=True)
full_df = full_df.reset_index(drop=True)

# --- Verification ---
print(f"--- Merge Complete ---")
print(f"Total shape: {full_df.shape}")
print(f"Date range: from {full_df['Datetime'].min()} to {full_df['Datetime'].max()}")

# Check for duplicates (optional but recommended when overlapping time ranges are possible)
duplicates = full_df.duplicated(subset=['Arc ID', 'Datetime']).sum()
print(f"Duplicate rows found (same ID and Time): {duplicates}")

#full_df.head()

--- Merge Complete ---
Total shape: (29882472, 15)
Date range: from 2024-10-01 03:00:00+00:00 to 2026-01-05 15:00:00+00:00
Duplicate rows found (same ID and Time): 0


## Uncomplete values analysis

Uncomplete data means either debit is missing or taux d'occupation is missing. 

**STEP 1:** We get a first overview of the uncomplete data

In [16]:
total_rows = len(full_df)

mask_debit_na = full_df['Debit'].isna()
mask_occ_na = full_df['Occupation'].isna()

# --- Calculations ---
count_any_missing = (mask_debit_na | mask_occ_na).sum()
count_only_debit_missing = (mask_debit_na & ~mask_occ_na).sum()
count_only_occ_missing = (mask_occ_na & ~mask_debit_na).sum()
count_both_missing = (mask_debit_na & mask_occ_na).sum()

# --- Displaying Percentages ---

print(f"Total data volume: {total_rows} rows\n")

print(f"At least one of the two values is missing (Total) : {count_any_missing / total_rows * 100:.2f}%")
print("-" * 60)
print(f"-> Of which only Debit is missing                 : {count_only_debit_missing / total_rows * 100:.2f}%")
print(f"-> Of which only Occupation is missing            : {count_only_occ_missing / total_rows * 100:.2f}%")
print(f"-> Of which BOTH are missing                      : {count_both_missing / total_rows * 100:.2f}%")

# Mathematical verification (the sum of the three sub-categories must equal the Total)
assert count_any_missing == (count_only_debit_missing + count_only_occ_missing + count_both_missing)

Total data volume: 29882472 rows

At least one of the two values is missing (Total) : 59.16%
------------------------------------------------------------
-> Of which only Debit is missing                 : 13.26%
-> Of which only Occupation is missing            : 10.68%
-> Of which BOTH are missing                      : 35.23%


**CANCELED** 
We retrieve and drop all the bad quality stations (giving at least 5% of uncomplete data)

In [17]:
# def get_low_quality_stations(df, threshold_percent):
#     """
#     Identifies stations ('Arc ID') that have a complete data rate below the given threshold.
#     A data point is "complete" if 'Debit' (Flow) AND 'Occupation' are not NaN.
    
#     Args:
#         df (pd.DataFrame): The dataset containing 'Arc ID', 'Debit', 'Occupation'.
#         threshold_percent (float): The minimum acceptable percentage (e.g., 95 for 95%).
        
#     Returns:
#         list: The list of 'Arc ID's that are below the threshold.
#     """
    
#     # 1. Create a boolean mask: True if the row is complete, False otherwise
#     # ~ means NOT, .isna() checks for NaNs
#     is_complete = ~df['Debit'].isna() & ~df['Occupation'].isna()
    
#     # 2. Group by station and calculate the percentage of True values
#     # The mean of booleans gives the ratio (e.g., 0.85). We multiply by 100.
#     station_completeness = is_complete.groupby(df['Arc ID']).mean() * 100
    
#     # 3. Filter stations below the threshold
#     bad_stations = station_completeness[station_completeness < threshold_percent]
    
#     # 4. Return the list of IDs (the index of the filtered series)
#     return bad_stations.index.tolist()

In [18]:
# threshold = 95 #value in percent

# # --- 1. Single calculation of scores per station ---
# # Calculate the percentage of valid data for EACH station only once
# is_complete = ~full_df['Debit'].isna() & ~full_df['Occupation'].isna()
# station_scores = is_complete.groupby(full_df['Arc ID']).mean() * 100

# total_stations = len(station_scores)

# # --- 2. Data generation for the plot ---
# thresholds = np.arange(0, 101, 1)
# kept_stations_ratios = []

# for t in thresholds:
#     count_valid = (station_scores >= t).sum()
    
#     ratio = (count_valid / total_stations) * 100
#     kept_stations_ratios.append(ratio)

# # --- 3. Plotting the curve ---
# plt.figure(figsize=(10, 6))
# plt.plot(thresholds, kept_stations_ratios, color='#007acc', linewidth=2.5)

# plt.axvline(x= threshold, color='red', linestyle='--', alpha=0.6, label='Quality Threshold')

# # Formatting
# plt.title("Percentage of stations with at least a given percentage of complete data", fontsize=14, fontweight='bold')
# plt.xlabel("Quality Requirement Threshold (%)", fontsize=12)
# plt.ylabel("% of Stations Retained", fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.7)
# plt.legend()
# plt.xlim(0, 100)
# plt.ylim(0, 105)

# plt.show()

In [19]:
# # --- Dropping data from bad stations ---

# bad_ids = get_low_quality_stations(full_df, threshold_percent=threshold)

# # 1. Store the initial size for comparison
# initial_rows = len(full_df)

# # 2. Identify the rows to keep
# # We keep rows where 'Arc ID' is NOT in the list of bad_ids.
# full_df_clean = full_df[~full_df['Arc ID'].isin(bad_ids)].copy()

# # 3. Calculate drop statistics
# dropped_rows = initial_rows - len(full_df_clean)
# dropped_stations = len(bad_ids)
# final_rows = len(full_df_clean)

# # 4. Display the summary
# print(f"--- Quality-based Removal Summary ---")
# print(f"Number of original stations: {len(full_df['Arc ID'].unique())}")
# print(f"Low-quality stations dropped: {dropped_stations}")
# print(f"Stations retained: {len(full_df_clean['Arc ID'].unique())}")
# print("-" * 40)
# print(f"Initial number of rows: {initial_rows}")
# print(f"Number of rows dropped: {dropped_rows}")
# print(f"Final number of rows: {final_rows}")

**STEP 3:** Once we dropped all the datapoints from bad quality stations, we drop the remaining bad quality data

In [20]:
# --- Dropping rows with missing traffic data ---

initial_rows = len(full_df)
full_df_clean = full_df.dropna(subset=['Debit', 'Occupation'])
final_rows = len(full_df_clean)

print(f"Initial number of rows: {initial_rows}")
print(f"Number of rows after cleaning: {final_rows}")
print(f"Rows dropped: {initial_rows - final_rows}")

Initial number of rows: 29882472
Number of rows after cleaning: 12203672
Rows dropped: 17678800


In [21]:
# --- 6. Exportation du dataset complet ---

# Définition du chemin de sortie complet
output_path = '/Users/quentingirard/Downloads/MCNDU_simple_clean_full_dataset.csv'

# Exportation en CSV
full_df_clean.to_csv(output_path, index=False, sep=';', encoding='utf-8')

print(f"✅ Dataset exporté avec succès ici : {output_path}")

✅ Dataset exporté avec succès ici : /Users/quentingirard/Downloads/MCNDU_simple_clean_full_dataset.csv
